# INSTRUCTOR NOTE: what from the lecture could go in this notebook
*Delete this cell before sharing with students.*

The Session 1 deck has 29 slides. Most are discussion slides that work better as slides. About a third are really notebook material: text that explains code, or a computation shown on a slide that students could run.

## A. Move in as text (no new code)

| Lecture slides | Put in the notebook as | Why |
|---|---|---|
| 20 · three forms, plus Grossman-Stiglitz | The "Market efficiency: three kinds" block below, with one line on Grossman-Stiglitz (information is costly, so markets are *mostly* efficient) | Frames both cells |
| 8 · joint hypothesis problem | A short box before Part A: every test of efficiency also assumes a model of expected returns. Part A assumes it is roughly constant | Explains why "no pattern" is weaker than it sounds |
| 25 · Part A results | "How to read this output" under cell 1: autocorrelation -0.039, p = 0.067, R² = 0.001, 2,262 days. Small, not significant at 5%, consistent with weak form but not proof | Replaces a results slide |
| 27 · Part B results | Same under cell 2, plus the "before you trade it" caveats: a few dozen events, no significance test, one stock, no trading costs | Replaces a results slide |
| 28 · discussion questions | A "Questions" section at the end, with the five questions | Students keep them with the code |

Moving the two results slides in saves about 10 minutes of Session 1.

## B. Add a small amount of code (same pattern as the existing cells)

| Lecture slides | New cell | Needs |
|---|---|---|
| 4-6 · CAPM, estimating beta, systematic vs. idiosyncratic risk | Jensen regression of Micron monthly excess returns on the market, in two sub-periods, to show beta drifting (slide 5 shows CVX moving from 0.58 to 0.81 in two years). Print R² as the systematic share of risk | Monthly prices and the Ken French market and risk-free series |
| 11-12 · Fama-French and Carhart, reading the output | The same regression with SMB, HML and UMD added; read the loadings and the R² data check ("a low R² for a fund often means a data error") | The Ken French factors |
| 10 · momentum (optional) | One chart of cumulative SMB, HML and UMD, to see the factors the lecture describes | The Ken French factors |

These are the parts of the lecture that currently have no live code behind them. Slide 12's CVX table comes from the textbook.

## C. Leave as slides

- Readings discussion: slides 15-19 (Blow-Up, Barron's, Global Alpha, Jacobs & Levy) and 21-23 (Fama 1998, Fama vs. Thaler, Mauboussin).
- Fama-French 1992 and Jegadeesh-Titman: the narrative needs the full stock universe, not one stock.
- DGTW, slides 13-14: needs CRSP and Compustat. The Session 3 notebook covers it, and it fits CK Chapter 3 in Session 2.
- WRDS ratio tips, slide 17: a good slide, not a computation.

## Suggested order for the combined notebook
1. Market efficiency: three kinds, and the joint hypothesis box
2. Part A (weak form), with its results text
3. Part B (semi-strong form), with its results text
4. CAPM and beta drift for Micron (new, B)
5. Fama-French and Carhart for Micron (new, B)
6. Discussion questions

## Known fixes to make at the same time
- The notes link ends in `#scrollTo=e0fd7678`, which opens at Part B. Link to the top instead.
- Part A ends at Jan 2024 but Part B runs to Oct 2026. Use one window.
- Cache the Yahoo data in Drive as a fallback (you plan to do this later).

# Market efficiency: three kinds

Eugene Fama's taxonomy asks one question: **what information is already in prices?** Each form includes the one before it.

| Form | Prices already reflect | So you should not be able to profit from | Tested below |
|---|---|---|---|
| **Weak** | All past price information: past prices and returns | Patterns in past prices, such as yesterday's return predicting today's | Cell 1: Micron's lag-1 autocorrelation and regression |
| **Semi-strong** | All public information (earnings, news, filings), absorbed essentially immediately | Trading on public news after it is released | Cell 2: Micron's price reaction to earnings surprises |
| **Strong** | All information, including private (insider) information | Even trading on inside information | Not tested here: it needs private information |

**What to look for**

- **Weak form:** an autocorrelation close to zero, a small and statistically insignificant coefficient, and an R² near zero.
- **Semi-strong form:** a jump at the announcement, in the direction of the surprise, is expected either way. The question is whether the price **keeps drifting** in that direction over the following days. That is post-earnings-announcement drift (Ball & Brown, 1968), a classic violation of semi-strong efficiency.

**Two cautions.** Finding no pattern does not prove markets are efficient. Finding one does not mean you could profit from it after costs.

In [2]:
import yfinance as yf
import pandas as pd
import numpy as np
import statsmodels.api as sm

# 1. Download Micron (MU) stock data from Yahoo Finance
ticker = 'MU'
# Using auto_adjust=True explicitly simplifies column structure to a single index
data = yf.download(ticker, start='2015-01-01', end='2024-01-01', auto_adjust=True)

# Handle potential MultiIndex columns returned by newer versions of yfinance
if isinstance(data.columns, pd.MultiIndex):
    data.columns = data.columns.get_level_values(0)

# 2. Calculate daily returns using 'Close' (which is adjusted because auto_adjust=True)
data['Returns'] = data['Close'].pct_change()
data = data.dropna()

# 3. Create the Lag-1 returns column
data['Lag1_Returns'] = data['Returns'].shift(1)
data = data.dropna()

# 4. Calculate Lag-1 Autocorrelation
lag1_autocorr = data['Returns'].corr(data['Lag1_Returns'])
print(f"Lag-1 Autocorrelation of daily returns: {lag1_autocorr:.4f}\n")

# 5. Perform OLS regression to see if Lag-1 predicts next day's return
# Dependent variable (Y): Today's Return
# Independent variable (X): Yesterday's Return (Lag-1)
Y = data['Returns']
X = sm.add_constant(data['Lag1_Returns'])

model = sm.OLS(Y, X).fit()
print(model.summary())


[*********************100%***********************]  1 of 1 completed

Lag-1 Autocorrelation of daily returns: -0.0385

                            OLS Regression Results                            
Dep. Variable:                Returns   R-squared:                       0.001
Model:                            OLS   Adj. R-squared:                  0.001
Method:                 Least Squares   F-statistic:                     3.358
Date:                Thu, 01 Oct 2026   Prob (F-statistic):             0.0670
Time:                        05:30:45   Log-Likelihood:                 4796.5
No. Observations:                2262   AIC:                            -9589.
Df Residuals:                    2260   BIC:                            -9577.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------

In [5]:
import yfinance as yf
import pandas as pd
import numpy as np

# 1. Fetch MU earnings history using yfinance ticker object
mu_ticker = yf.Ticker('MU')

# Get historical earnings dates and surprises
earnings = mu_ticker.get_earnings_dates(limit=50)
if earnings is not None and not earnings.empty:
    # Clean the index and columns
    earnings = earnings.reset_index()
    earnings['Earnings Date'] = pd.to_datetime(earnings['Earnings Date']).dt.tz_localize(None)

    # Drop rows without actual or estimated EPS
    earnings = earnings.dropna(subset=['EPS Estimate', 'Reported EPS'])

    # Calculate raw surprise amount
    earnings['Surprise'] = earnings['Reported EPS'] - earnings['EPS Estimate']

    # Get daily price data over a wide window
    price_data = yf.download('MU', start='2014-01-01', end='2026-10-01', auto_adjust=True)
    if isinstance(price_data.columns, pd.MultiIndex):
        price_data.columns = price_data.columns.get_level_values(0)

    results_list = []

    for idx, row in earnings.iterrows():
        e_date = row['Earnings Date']
        # Find closest trading day on or after the earnings date
        trading_days = price_data.index[price_data.index >= e_date]
        if len(trading_days) < 5:
            continue

        t0 = trading_days[0] # Day of/immediately after earnings announcement
        # Find the trading day before earnings
        days_before = price_data.index[price_data.index < e_date]
        if len(days_before) == 0:
            continue
        t_minus_1 = days_before[-1]

        # Calculate immediate price jump (Open of t0 vs Close of t-1)
        open_t0 = price_data.loc[t0, 'Open']
        close_tm1 = price_data.loc[t_minus_1, 'Close']
        immediate_jump = (open_t0 - close_tm1) / close_tm1

        # Calculate 3-day post-earnings holding return (Close of t+3 vs Close of t0)
        t3 = trading_days[3]
        close_t0 = price_data.loc[t0, 'Close']
        close_t3 = price_data.loc[t3, 'Close']
        post_return = (close_t3 - close_t0) / close_t0

        results_list.append({
            'Earnings_Date': e_date.strftime('%Y-%m-%d'),
            'Reported_EPS': row['Reported EPS'],
            'EPS_Estimate': row['EPS Estimate'],
            'Surprise': row['Surprise'],
            'Immediate_Jump': immediate_jump,
            'Post_3D_Return': post_return
        })

    results_df = pd.DataFrame(results_list)

    # Classify surprises
    positive_surprises = results_df[results_df['Surprise'] > 0]
    negative_surprises = results_df[results_df['Surprise'] <= 0]

    print("=== Micron (MU) Earnings Surprise Impact Analysis ===\n")
    print(f"Average immediate jump after POSITIVE surprises: {positive_surprises['Immediate_Jump'].mean():.2%}")
    print(f"Average 3-day post return after POSITIVE surprises: {positive_surprises['Post_3D_Return'].mean():.2%}\n")
    print(f"Average immediate jump after NEGATIVE surprises: {negative_surprises['Immediate_Jump'].mean():.2%}")
    print(f"Average 3-day post return after NEGATIVE surprises: {negative_surprises['Post_3D_Return'].mean():.2%}\n")

    print("--- Sample Earnings Events Details ---")
    print(results_df.head(10).to_string(index=False))
else:
    print("Earnings history data is currently unavailable via Yahoo Finance API for this ticker.")

[*********************100%***********************]  1 of 1 completed

=== Micron (MU) Earnings Surprise Impact Analysis ===

Average immediate jump after POSITIVE surprises: 2.30%
Average 3-day post return after POSITIVE surprises: 0.09%

Average immediate jump after NEGATIVE surprises: -5.84%
Average 3-day post return after NEGATIVE surprises: -4.72%

--- Sample Earnings Events Details ---
Earnings_Date  Reported_EPS  EPS_Estimate  Surprise  Immediate_Jump  Post_3D_Return
   2026-06-24         25.11         20.69      4.42        0.175954       -0.048840
   2026-03-18         12.20          9.16      3.04       -0.079614       -0.109708
   2025-12-17          4.78          3.96      0.82        0.137504        0.111527
   2025-09-23          3.03          2.86      0.17       -0.007992        0.013543
   2025-06-25          1.91          1.59      0.32        0.017917       -0.040556
   2025-03-20          1.56          1.42      0.14       -0.046602       -0.027344
   2024-12-18          1.79          1.77      0.02       -0.133013        0.025146
   2